# Segmentation d'images avec Keras

Détourer les forêts sur des photographies aériennes de l'IGN : c'est un problème de segmentation, où le modèle classe chaque pixel, forêt ou non-forêt. Ce notebook entraîne et compare trois architectures, présentées dans les slides « Segmentation et UNet » :

1. un UNet simple, entraîné depuis zéro ;
2. un encodeur EfficientNetV2 pré-entraîné sur ImageNet, suivi d'un décodeur sans connexions de saut : ce qu'on perd sans elles ;
3. un UNet dont l'encodeur est ce même EfficientNetV2, d'abord gelé, puis fine-tuné.

Il demande un GPU : *Exécution > Modifier le type d'exécution > GPU T4*. Le téléchargement des données (environ 0,8 Go) prend quelques minutes : lancez les trois cellules suivantes dès le début.

In [ ]:
import math
import random

import keras
import matplotlib.pyplot as plt
import numpy
import pandas
from imageio.v3 import imread
from keras import layers

In [ ]:
!nvidia-smi

In [ ]:
%%time
!rm -rf sample_data
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh ign-forest-2009

## Chargement des données

Les données sont des photographies aériennes prises par l'IGN en 2009 en Lorraine : 20 zones de 5 km × 5 km. Chaque zone est découpée en 9 × 9 = 81 tuiles de 1024 × 1024 pixels (1 pixel = 50 cm sur le terrain : une tuile couvre 512 m × 512 m).

- **Entrée** (fichiers `*_irc.jpg`) : une image en *infrarouge couleur* (IRC). Ses trois canaux sont le proche infrarouge, le rouge et le vert, affichés en rouge, vert et bleu. La végétation réfléchit fortement l'infrarouge : elle apparaît en rouge.
- **Cible** (fichiers `*_pfnf_nch.png`) : un masque en noir et blanc, la classe de chaque pixel : 1 pour forêt, 0 pour non-forêt. Au sens de l'IGN, une forêt n'est pas un simple groupe d'arbres : c'est une surface de plus de 5 000 m², large d'au moins 20 m, dont la végétation couvre au moins 10 % du sol et qui n'est pas utilisée pour l'agriculture.

2 zones ont été mises de côté pour la validation, et 2 autres pour le test. Toutes les tuiles d'une zone vont dans le même ensemble : des tuiles voisines se ressemblent trop pour mesurer honnêtement la généralisation. Il reste 16 zones pour l'entraînement, soit 1 296 tuiles.

In [ ]:
df_entrainement = pandas.read_csv("ign-forest-2009/train-files.csv")
df_val = pandas.read_csv("ign-forest-2009/val-files.csv")
df_test = pandas.read_csv("ign-forest-2009/test-files.csv")

print("Coordonnées en entraînement")
print(df_entrainement.coord.unique())

print("Coordonnées en val")
print(df_val.coord.unique())

print("Coordonnées en test")
print(df_test.coord.unique())

df_test

## Étude d'un exemple et de son annotation

In [ ]:
exemple_entree = imread(df_entrainement.Observations[0])
exemple_annotation = imread(df_entrainement.Annotations[0])

In [ ]:
def affiche_echantillon(*images: numpy.ndarray) -> None:
  """Affiche côte à côte une image d'entrée, sa cible et la prédiction."""
  figure, axes = plt.subplots(1, len(images), figsize=(18, 18))

  titres = ["Entrée", "Cible", "Prédiction"]
  for image, ax, titre in zip(images, axes, titres):
    image = numpy.asarray(image)
    ax.set_title(titre)
    if image.ndim == 3 and image.shape[2] == 3:
      # Image en infrarouge couleur, pixels de 0 à 255
      ax.imshow(image.astype("uint8"))
    else:
      if image.ndim == 3:
        # Cible d'un batch : son seul canal
        image = image[..., -1]
      # Bornes fixes : forêt (1) en jaune, non-forêt (0) en violet, même quand
      # toute la tuile est d'une seule classe
      ax.imshow(image, vmin=0, vmax=1)
    ax.axis("off")
  plt.show()


affiche_echantillon(exemple_entree, exemple_annotation)

## Options générales

In [ ]:
# Dimension des images originales
dim_originale = 1024

# Dimension des images en entrée du réseau
dim_entree = 512

# Nombre de canaux des images en entrée
nb_canaux = 3

# Nombre de fils d'exécution qui préparent les batchs pendant l'entraînement
nb_fils = 2

# Nombre d'epochs du UNet entraîné depuis zéro, des modèles à encodeur
# pré-entraîné (gelé, seul leur décodeur apprend : ils convergent plus vite), puis
# du fine-tuning
nb_epoques = 15
nb_epoques_pretraine = 5
nb_epoques_affinage = 3

## Augmentation des données

Avec 1 296 tuiles d'entraînement, le réseau reverrait vite les mêmes images : on les augmente par des transformations aléatoires, qui le rendent moins sensible aux rotations, au cadrage, etc. Implémentez la fonction `transforme_donnees` :

- si `evaluation` vaut `True` (validation et test), un recadrage centré de `X` et de `Y` à la taille `taille_recadree` ;
- sinon (entraînement), un recadrage aléatoire à cette taille, puis une rotation d'un nombre aléatoire de quarts de tour et des miroirs aléatoires.

Écrivez-la en NumPy, sans TensorFlow : `PyDataset` l'appelle depuis plusieurs fils d'exécution (`nb_fils`), qui préparent les batchs sur le CPU pendant que le GPU entraîne le modèle. La cible doit subir exactement la même transformation que l'image, sinon le masque ne correspond plus aux pixels. Vérifiez-le sur l'exemple, en décommentant la dernière ligne de la cellule.

**Documentation utile** :

- [Découpage des tableaux NumPy](https://numpy.org/doc/stable/user/basics.indexing.html#slicing-and-striding)
- [`numpy.rot90`](https://numpy.org/doc/stable/reference/generated/numpy.rot90.html)
- [`numpy.fliplr`](https://numpy.org/doc/stable/reference/generated/numpy.fliplr.html)
- [`numpy.flipud`](https://numpy.org/doc/stable/reference/generated/numpy.flipud.html)
- [`random.randint`](https://docs.python.org/3/library/random.html#random.randint)

In [ ]:
def transforme_donnees(
  X: numpy.ndarray,
  Y: numpy.ndarray,
  evaluation: bool = False,
  taille_recadree: tuple[int, int] = (dim_entree, dim_entree),
) -> tuple[numpy.ndarray, numpy.ndarray]:
  # Votre code ici
  return X, Y


# affiche_echantillon(*transforme_donnees(exemple_entree, exemple_annotation))

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `hauteur, largeur = X.shape[:2]` : les tuiles font 1 024 × 1 024 pixels, et `taille_recadree` vaut (512, 512)
- `X[haut : haut + hauteur_recadree, gauche : gauche + largeur_recadree]` : le recadrage, de coin en haut à gauche (`haut`, `gauche`)
- Recadrage centré : `haut = (hauteur - hauteur_recadree) // 2` ; aléatoire : `random.randint(0, hauteur - hauteur_recadree)`, bornes comprises
- `numpy.rot90(X, k)` : `k` quarts de tour dans le plan des deux premiers axes, avec `k = random.randint(0, 3)`
- `numpy.fliplr(X)` et `numpy.flipud(X)`, chacun une fois sur deux : `if random.random() < 0.5:`
- Tirez chaque paramètre aléatoire une seule fois, puis appliquez-le à `X` et à `Y`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def transforme_donnees(
  X: numpy.ndarray,
  Y: numpy.ndarray,
  evaluation: bool = False,
  taille_recadree: tuple[int, int] = (dim_entree, dim_entree),
) -> tuple[numpy.ndarray, numpy.ndarray]:
  hauteur, largeur = X.shape[:2]
  hauteur_recadree, largeur_recadree = taille_recadree
  if evaluation:
    haut, gauche = ..., ...  # recadrage centré
  else:
    haut = random.randint(0, ...)
    gauche = random.randint(0, ...)
  X = X[haut : haut + hauteur_recadree, gauche : gauche + largeur_recadree]
  Y = Y[haut : haut + hauteur_recadree, gauche : gauche + largeur_recadree]
  if not evaluation:
    k = random.randint(0, 3)  # nombre de quarts de tour
    X, Y = numpy.rot90(X, k), numpy.rot90(Y, k)
    if random.random() < 0.5:
      X, Y = ..., ...  # miroir gauche-droite
    if random.random() < 0.5:
      X, Y = ..., ...  # miroir haut-bas
  return X, Y
```

### Solution

In [ ]:
def transforme_donnees(
  X: numpy.ndarray,
  Y: numpy.ndarray,
  evaluation: bool = False,
  taille_recadree: tuple[int, int] = (dim_entree, dim_entree),
) -> tuple[numpy.ndarray, numpy.ndarray]:
  hauteur, largeur = X.shape[:2]
  hauteur_recadree, largeur_recadree = taille_recadree
  if evaluation:
    # Recadrage centré
    haut = (hauteur - hauteur_recadree) // 2
    gauche = (largeur - largeur_recadree) // 2
  else:
    # Recadrage aléatoire
    haut = random.randint(0, hauteur - hauteur_recadree)
    gauche = random.randint(0, largeur - largeur_recadree)
  X = X[haut : haut + hauteur_recadree, gauche : gauche + largeur_recadree]
  Y = Y[haut : haut + hauteur_recadree, gauche : gauche + largeur_recadree]
  if not evaluation:
    # Rotation d'un nombre aléatoire de quarts de tour
    k = random.randint(0, 3)
    X, Y = numpy.rot90(X, k), numpy.rot90(Y, k)
    # Miroirs aléatoires, la même transformation pour l'image et la cible
    if random.random() < 0.5:
      X, Y = numpy.fliplr(X), numpy.fliplr(Y)
    if random.random() < 0.5:
      X, Y = numpy.flipud(X), numpy.flipud(Y)
  return X, Y


for _ in range(3):
  affiche_echantillon(*transforme_donnees(exemple_entree, exemple_annotation))

## Création d'un jeu de données Keras

Keras lit les données d'entraînement batch par batch, dans un jeu de données : un `keras.utils.PyDataset`. Écrivez la classe `DatasetTuiles`, qui en hérite :

- `__init__` garde les chemins des images et des cibles, la taille des batchs, le mode (`evaluation`), et l'ordre de lecture des tuiles : mélangé pour l'entraînement, dans l'ordre sinon. Il transmet ses autres arguments (`**kwargs`, comme `nb_fils`) au constructeur de `PyDataset` ;
- `__len__` renvoie le nombre de batchs par epoch ;
- `__getitem__(index)` lit les tuiles du batch numéro `index`, les transforme avec `transforme_donnees`, et renvoie deux tableaux NumPy de `float32` : les images, de forme (batch, 512, 512, 3), et les cibles, de forme (batch, 512, 512, 1). La cible vaut 1 pour la forêt, 0 sinon ;
- `on_epoch_end`, appelée à la fin de chaque epoch, remélange les tuiles d'entraînement.

Les pixels restent entre 0 et 255 : chaque modèle les normalise lui-même.

**Documentation utile** :

- [`keras.utils.PyDataset`](https://keras.io/api/utils/python_utils/#pydataset-class)
- [`numpy.random.permutation`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.permutation.html)
- [`numpy.stack`](https://numpy.org/doc/stable/reference/generated/numpy.stack.html)
- [`numpy.ndarray.astype`](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.astype.html)

In [ ]:
class DatasetTuiles(keras.utils.PyDataset):
  """Fournit des batchs de tuiles et de cibles, transformées."""

  def __init__(
    self,
    df: pandas.DataFrame,
    taille_lot: int,
    evaluation: bool,
    taille_recadree: tuple[int, int] = (dim_entree, dim_entree),
    **kwargs,
  ) -> None:
    """Initialise le jeu de données."""
    super().__init__(**kwargs)
    # Votre code ici

  def __len__(self) -> int:
    """Calcule le nombre de batchs par epoch."""
    # Votre code ici

  def __getitem__(self, index: int) -> tuple[numpy.ndarray, numpy.ndarray]:
    """Renvoie le batch numéro `index`."""
    # Votre code ici

  def on_epoch_end(self) -> None:
    """Remélange les tuiles d'entraînement après chaque epoch."""
    # Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `df.Observations.to_numpy()` et `df.Annotations.to_numpy()` : les chemins des images et des cibles, numérotés à partir de 0, même quand `df` est un extrait d'un autre tableau
- `numpy.arange(n)` : les tuiles dans l'ordre, pour la validation et le test ; `numpy.random.permutation(n)` : dans un ordre aléatoire, pour l'entraînement
- `math.ceil(n / self.taille_lot)` batchs : le dernier peut être incomplet
- Les tuiles du batch : `self.indices[debut : debut + self.taille_lot]`, avec `debut = index * self.taille_lot`
- `imread(chemin)` lit une tuile ou sa cible ; `imread(chemin)[..., numpy.newaxis]` : la cible avec un axe de canal, de forme (1 024, 1 024, 1), 1 pour la forêt
- `numpy.stack(Xs).astype("float32")` : une liste de tableaux de même forme, empilés en un batch de `float32`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
class DatasetTuiles(keras.utils.PyDataset):
  def __init__(
    self,
    df: pandas.DataFrame,
    taille_lot: int,
    evaluation: bool,
    taille_recadree: tuple[int, int] = (dim_entree, dim_entree),
    **kwargs,
  ) -> None:
    super().__init__(**kwargs)
    self.chemins_X = df.Observations.to_numpy()
    self.chemins_Y = df.Annotations.to_numpy()
    self.taille_lot = taille_lot
    self.evaluation = evaluation
    self.taille_recadree = taille_recadree
    n = len(self.chemins_X)
    self.indices = numpy.arange(n) if evaluation else ...

  def __len__(self) -> int:
    return ...

  def __getitem__(self, index: int) -> tuple[numpy.ndarray, numpy.ndarray]:
    debut = index * self.taille_lot
    Xs, Ys = [], []
    for i in self.indices[debut : debut + self.taille_lot]:
      X = imread(self.chemins_X[i])
      Y = ...  # cible, de forme (1 024, 1 024, 1)
      X, Y = transforme_donnees(X, Y, self.evaluation, self.taille_recadree)
      Xs.append(X)
      Ys.append(Y)
    return ..., ...

  def on_epoch_end(self) -> None:
    if not self.evaluation:
      self.indices = ...
```

### Solution

In [ ]:
class DatasetTuiles(keras.utils.PyDataset):
  """Fournit des batchs de tuiles et de cibles, transformées."""

  def __init__(
    self,
    df: pandas.DataFrame,
    taille_lot: int,
    evaluation: bool,
    taille_recadree: tuple[int, int] = (dim_entree, dim_entree),
    **kwargs,
  ) -> None:
    """Initialise le jeu de données."""
    super().__init__(**kwargs)
    self.chemins_X = df.Observations.to_numpy()
    self.chemins_Y = df.Annotations.to_numpy()
    assert self.chemins_X.shape[0] == self.chemins_Y.shape[0]
    self.taille_lot = taille_lot
    self.evaluation = evaluation
    self.taille_recadree = taille_recadree
    n = len(self.chemins_X)
    # Ordre de lecture des tuiles : mélangé pour l'entraînement
    self.indices = numpy.arange(n) if evaluation else numpy.random.permutation(n)

  def __len__(self) -> int:
    """Calcule le nombre de batchs par epoch."""
    return math.ceil(len(self.chemins_X) / self.taille_lot)

  def __getitem__(self, index: int) -> tuple[numpy.ndarray, numpy.ndarray]:
    """Renvoie le batch numéro `index`."""
    debut = index * self.taille_lot
    Xs, Ys = [], []
    for i in self.indices[debut : debut + self.taille_lot]:
      X = imread(self.chemins_X[i])
      # Classe de chaque pixel (1 pour forêt), avec un axe de canal
      # pour subir les mêmes transformations que l'image
      Y = imread(self.chemins_Y[i])[..., numpy.newaxis]
      X, Y = transforme_donnees(X, Y, self.evaluation, self.taille_recadree)
      Xs.append(X)
      Ys.append(Y)
    return numpy.stack(Xs).astype("float32"), numpy.stack(Ys).astype("float32")

  def on_epoch_end(self) -> None:
    """Remélange les tuiles d'entraînement après chaque epoch."""
    if not self.evaluation:
      self.indices = numpy.random.permutation(len(self.chemins_X))


dataset = DatasetTuiles(df_entrainement, 3, evaluation=True)
X, Y = dataset[0]
X.shape, Y.shape

## Création du modèle

Le modèle passe d'un batch d'images de forme (batch, 512, 512, 3) à une carte de même taille, de forme (batch, 512, 512, 1) : la probabilité que chaque pixel soit de la forêt. Écrivez `construit_modele`, qui construit un UNet avec l'API fonctionnelle, comme dans les slides « Un UNet simple avec Keras » et « Passe avant du UNet », mais plus profond :

- `nb_niveaux` niveaux, avec `profondeur_initiale` filtres au premier, puis deux fois plus à chaque niveau, sans dépasser 512 : 32, 64, 128, 256 et 512 par défaut ;
- à chaque niveau, un `bloc_convolution` plus riche que celui des slides : deux convolutions 3 × 3, chacune suivie d'une normalisation par batch et d'une ReLU, puis un *dropout* de 10 % ;
- l'encodeur divise la résolution par deux d'un niveau au suivant, de 512 à 32 pixels de côté ; le décodeur la double à chaque niveau, et concatène la sortie de l'encodeur de même résolution (connexion de saut) ;
- en entrée, `Rescaling` ramène les pixels entre 0 et 1 ; en sortie, une convolution 1 × 1 à un seul filtre, avec une activation `sigmoid`.

Cette architecture vient de [l'article qui présente U-Net](https://arxiv.org/abs/1505.04597).

In [ ]:
def construit_modele(profondeur_initiale: int = 32, nb_niveaux: int = 5) -> keras.Model:
  # Votre code ici
  ...

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `profondeurs = [min(512, profondeur_initiale * 2**i) for i in range(nb_niveaux)]` : le nombre de filtres de chaque niveau
- `layers.Conv2D(filtres, 3, padding="same", use_bias=False)` : `padding="same"` garde la taille de l'image ; le biais est inutile avant une normalisation par batch, qui recentre les sorties
- `layers.BatchNormalization()`, `layers.ReLU()` et `layers.Dropout(0.1)`
- `layers.MaxPooling2D(2)` divise la résolution par deux, `layers.Conv2DTranspose(filtres, 2, strides=2)` la double
- `layers.Concatenate()([x, saut])` : la connexion de saut, sur l'axe des canaux
- `layers.Conv2D(1, 1, activation="sigmoid")` : la couche de sortie
- `construit_modele().output_shape` doit valoir `(None, 512, 512, 1)`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def bloc_convolution(x, filtres: int):
  for _ in range(2):
    x = layers.Conv2D(filtres, 3, padding="same", use_bias=False)(x)
    x = ...  # normalisation par batch
    x = ...  # ReLU
  return layers.Dropout(0.1)(x)


def construit_modele(profondeur_initiale: int = 32, nb_niveaux: int = 5) -> keras.Model:
  profondeurs = [min(512, profondeur_initiale * 2**i) for i in range(nb_niveaux)]
  entrees = keras.Input(shape=(dim_entree, dim_entree, nb_canaux))
  x = layers.Rescaling(1 / 255)(entrees)
  sauts = []
  for profondeur in profondeurs[:-1]:  # encodeur
    x = bloc_convolution(x, profondeur)
    sauts.append(x)
    x = ...  # résolution divisée par deux
  x = bloc_convolution(x, profondeurs[-1])  # fond du « U »
  for profondeur, saut in zip(reversed(profondeurs[:-1]), reversed(sauts)):  # décodeur
    x = ...  # résolution doublée
    x = bloc_convolution(..., profondeur)  # connexion de saut
  sorties = ...
  return keras.Model(entrees, sorties, name="unet")
```

### Solution

In [ ]:
# Le modèle U-Net de l'article https://arxiv.org/abs/1505.04597 (voir aussi
# les slides sur le UNet)


def bloc_convolution(x, filtres: int):
  """Deux convolutions 3 × 3 avec normalisation par batch et ReLU, puis un dropout."""
  for _ in range(2):
    x = layers.Conv2D(filtres, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
  return layers.Dropout(0.1)(x)


def construit_modele(profondeur_initiale: int = 32, nb_niveaux: int = 5) -> keras.Model:
  profondeurs = [min(512, profondeur_initiale * 2**i) for i in range(nb_niveaux)]
  # Pixels de 0 à 255
  entrees = keras.Input(shape=(dim_entree, dim_entree, nb_canaux))
  x = layers.Rescaling(1 / 255)(entrees)

  # Encodeur : la résolution est divisée par deux d'un niveau au suivant, et
  # la sortie de chaque niveau est gardée pour la connexion de saut
  sauts = []
  for profondeur in profondeurs[:-1]:
    x = bloc_convolution(x, profondeur)
    sauts.append(x)
    x = layers.MaxPooling2D(2)(x)
  x = bloc_convolution(x, profondeurs[-1])  # fond du « U », 32 × 32

  # Décodeur : la résolution est doublée, puis la sortie de l'encodeur de même
  # résolution est concaténée (connexion de saut)
  for profondeur, saut in zip(reversed(profondeurs[:-1]), reversed(sauts)):
    x = layers.Conv2DTranspose(profondeur, 2, strides=2)(x)
    x = bloc_convolution(layers.Concatenate()([x, saut]), profondeur)

  # Probabilité que chaque pixel soit de la forêt
  sorties = layers.Conv2D(1, 1, activation="sigmoid")(x)
  return keras.Model(entrees, sorties, name="unet")


construit_modele().output_shape

## Entraînement

La cellule suivante crée les jeux de données d'entraînement et de validation, qui lisent des batchs de 4 tuiles.

On suit l'exactitude (*accuracy*), la part des pixels bien classés, et l'IoU de la classe forêt (slide « Pertes et métriques en segmentation ») : la surface où la prédiction et la cible disent toutes deux « forêt », divisée par la surface où au moins l'une des deux le dit. Elle vaut 1 pour un masque parfait, et 0 pour un modèle qui ne prédit jamais la forêt, quelle que soit son exactitude : c'est sur elle qu'on choisit le meilleur modèle.

Le notebook entraîne quatre modèles de la même façon. Écrivez la fonction `entraine(modele, taux_apprentissage, chemin_sauvegarde, nb_epoques)`, sur le modèle de la slide « Entraîner un UNet : perte et IoU ». Elle :

- compile `modele` avec l'optimiseur Adam au pas `taux_apprentissage`, l'entropie croisée binaire comme perte, l'exactitude et l'IoU de la forêt comme métriques ;
- l'entraîne pendant `nb_epoques` epochs sur `donnees_entrainement`, en l'évaluant sur `donnees_val` à la fin de chaque epoch ;
- arrête l'entraînement quand la perte de validation n'a pas baissé depuis 5 epochs, et sauvegarde dans le fichier `chemin_sauvegarde` le modèle de meilleure `val_iou` ;
- recharge dans `modele` ces meilleurs poids, puis renvoie l'historique de l'entraînement.

Entraînez ensuite le UNet de `construit_modele` avec un pas de 1e-4, en le sauvegardant dans `meilleur_unet.keras`. Keras affiche la durée de chaque epoch.

In [ ]:
taille_lot = 4

# Jeux de données d'entraînement et de validation : `workers` fils d'exécution
# préparent les batchs suivants pendant que le GPU calcule
donnees_entrainement = DatasetTuiles(
  df_entrainement, taille_lot, evaluation=False, workers=nb_fils
)
donnees_val = DatasetTuiles(df_val, taille_lot, evaluation=True, workers=nb_fils)

# Scores de test de chaque modèle, comparés à la fin du notebook
resultats_test: dict[str, dict[str, float]] = {}

In [ ]:
def entraine(
  modele: keras.Model,
  taux_apprentissage: float,
  chemin_sauvegarde: str,
  nb_epoques: int = nb_epoques,
) -> keras.callbacks.History:
  # Votre code ici
  ...


# Votre code ici : le UNet de construit_modele, entraîné avec un pas de 1e-4

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `keras.metrics.BinaryIoU(target_class_ids=[1], threshold=0.5, name="iou")` : l'IoU de la classe 1 (forêt), une fois les probabilités seuillées à 0,5 ; `name="iou"` donne son nom à `val_iou`
- `modele.compile(optimizer=keras.optimizers.Adam(taux_apprentissage), loss="binary_crossentropy", metrics=["accuracy", iou])`
- `keras.callbacks.EarlyStopping(monitor="val_loss", patience=5)`
- `keras.callbacks.ModelCheckpoint(chemin_sauvegarde, monitor="val_iou", mode="max", save_best_only=True)` : `mode="max"`, puisqu'une IoU plus grande est meilleure
- `modele.fit(...)` renvoie l'historique ; `modele.load_weights(chemin_sauvegarde)` recharge les poids sauvegardés
- `modele.summary()` affiche les couches du modèle et son nombre de paramètres

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def entraine(
  modele: keras.Model,
  taux_apprentissage: float,
  chemin_sauvegarde: str,
  nb_epoques: int = nb_epoques,
) -> keras.callbacks.History:
  iou = keras.metrics.BinaryIoU(target_class_ids=[1], threshold=0.5, name="iou")
  modele.compile(
    optimizer=...,
    loss="binary_crossentropy",
    metrics=["accuracy", iou],
  )
  historique = modele.fit(
    donnees_entrainement,
    validation_data=...,
    epochs=nb_epoques,
    callbacks=[
      keras.callbacks.EarlyStopping(monitor="val_loss", patience=5),
      keras.callbacks.ModelCheckpoint(
        chemin_sauvegarde, monitor=..., mode=..., save_best_only=True
      ),
    ],
  )
  modele.load_weights(chemin_sauvegarde)
  return historique


modele = construit_modele()
modele.summary()
historique = entraine(modele, ..., "meilleur_unet.keras")
```

### Solution

In [ ]:
def entraine(
  modele: keras.Model,
  taux_apprentissage: float,
  chemin_sauvegarde: str,
  nb_epoques: int = nb_epoques,
) -> keras.callbacks.History:
  """Entraîne `modele`, puis recharge ses poids de meilleure IoU de validation."""
  # IoU de la classe forêt (1), les probabilités étant seuillées à 0,5
  iou = keras.metrics.BinaryIoU(target_class_ids=[1], threshold=0.5, name="iou")
  # Un nouveau compile à chaque appel : il prend en compte les couches gelées
  # ou dégelées depuis le précédent
  modele.compile(
    optimizer=keras.optimizers.Adam(taux_apprentissage),
    loss="binary_crossentropy",
    metrics=["accuracy", iou],
  )
  historique = modele.fit(
    donnees_entrainement,
    validation_data=donnees_val,
    epochs=nb_epoques,
    callbacks=[
      keras.callbacks.EarlyStopping(monitor="val_loss", patience=5),
      keras.callbacks.ModelCheckpoint(
        chemin_sauvegarde, monitor="val_iou", mode="max", save_best_only=True
      ),
    ],
  )
  modele.load_weights(chemin_sauvegarde)
  return historique


modele = construit_modele()
modele.summary()
historique = entraine(modele, 1e-4, "meilleur_unet.keras")

## Application aux données de test

La cellule suivante crée le jeu de données de test, et place dans `X` et `Y` 8 tuiles de test qui ont des lisières : entre 20 et 80 % de forêt dans la partie recadrée que voit le modèle.

Évaluez le modèle sur tout le jeu de test, et rangez ses scores dans `resultats_test["UNet entraîné depuis zéro"]`, sous la forme d'un dictionnaire. Affichez ensuite ses prédictions pour les 8 tuiles de `X` avec `affiche_echantillon`.

Les prédictions sont-elles de bonne qualité ? Comparez l'IoU de test à celle d'un modèle qui ne prédirait jamais la forêt, et regardez où se trouvent les erreurs.

**Documentation utile** :

- [`Model.evaluate`](https://keras.io/api/models/model_training_apis/#evaluate-method)

In [ ]:
donnees_test = DatasetTuiles(df_test, 8, evaluation=True, workers=nb_fils)

# 8 tuiles de test qui ont des lisières : entre 20 et 80 % de forêt dans la partie
# recadrée que voit le modèle
recadrage = (dim_originale - dim_entree) // 2


def calcule_part_foret(chemin: str) -> float:
  foret = imread(chemin)
  return foret[
    recadrage : recadrage + dim_entree, recadrage : recadrage + dim_entree
  ].mean()


parts = df_test.Annotations.map(calcule_part_foret)
df_lisieres = df_test[parts.between(0.2, 0.8)].sample(8, random_state=0)
X, Y = DatasetTuiles(df_lisieres, 8, evaluation=True)[0]
X.shape, Y.shape

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `modele.evaluate(donnees_test, return_dict=True)` : un dictionnaire `{"accuracy": …, "iou": …, "loss": …}`, au lieu d'une liste de valeurs
- `modele.predict(X)` : les probabilités de forêt, de forme (8, 512, 512, 1) ; `[..., 0]` retire le dernier axe
- `for x, y, p in zip(X, Y, predictions):` parcourt les images, les cibles et les prédictions ensemble

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
scores = modele.evaluate(...)
resultats_test["UNet entraîné depuis zéro"] = scores
print(resultats_test)
predictions = ...[..., 0]
for x, y, p in zip(X, Y, predictions):
  affiche_echantillon(x, y, p)
```

### Solution

In [ ]:
scores = modele.evaluate(donnees_test, return_dict=True)
resultats_test["UNet entraîné depuis zéro"] = scores
print(resultats_test)

# Probabilités de forêt de chaque pixel des 8 tuiles
predictions = modele.predict(X)[..., 0]
for x, y, p in zip(X, Y, predictions):
  affiche_echantillon(x, y, p)

Un modèle qui ne prédit jamais la forêt a une IoU nulle, quelle que soit son exactitude. Celui qui répond toujours la classe majoritaire, ici la forêt, a une exactitude et une IoU égales à la part de forêt des tuiles de test, autour de 0,6 (la comparaison des modèles, en fin de TP, calcule cette part) : l'IoU de test du UNet est loin de l'une comme de l'autre, et les grandes zones de forêt et de non-forêt sont bien détourées. Les erreurs se concentrent :

- le long des lisières, où la prédiction hésite (probabilités intermédiaires) et où la cible elle-même n'est précise qu'à quelques pixels près ;
- sur les petits objets, haies, bosquets, clairières ou chemins forestiers, que la définition de l'IGN (plus de 5 000 m², plus de 20 m de large) ne classe pas toujours comme l'œil ;
- sur la végétation qui n'est pas de la forêt, prairies et cultures, rouge elle aussi en infrarouge couleur.

## Modèle pré-entraîné sans connexions de saut

Le UNet précédent apprend tout depuis zéro, encodeur compris, sur 1 296 tuiles. Un encodeur pré-entraîné sur ImageNet sait déjà extraire des bords, des textures et des formes : c'est l'apprentissage par transfert (slides « Transfer learning »). ImageNet ne contient pas d'images infrarouges, mais ces caractéristiques visuelles restent utiles.

La cellule suivante charge EfficientNetV2B0 sans sa tête de classification, et le gèle : seul le décodeur apprendra. Pour une tuile de 512 × 512, il rend des cartes de 16 × 16 × 1 280. EfficientNetV2 normalise lui-même ses entrées : il reçoit les pixels entre 0 et 255, sans `Rescaling`.

Écrivez un décodeur volontairement simple, **sans connexions de saut** : il ne fait que suréchantillonner la sortie de l'encodeur, de 16 × 16 jusqu'à 512 × 512. Ce modèle montre ce qu'on perd sans elles : l'encodeur reconnaît la forêt, mais les détails fins calculés par ses premières couches n'atteignent jamais le décodeur.

Entraînez-le avec `entraine`, avec un pas de 1e-3 (seul le décodeur, parti de zéro, apprend) pendant `nb_epoques_pretraine` epochs, en le sauvegardant dans `meilleur_sans_saut.keras`. Rangez ses scores de test dans `resultats_test["EfficientNetV2B0 sans connexions de saut"]`, et comparez ses prédictions sur `X` à celles du UNet.

**Documentation utile** :

- [`layers.UpSampling2D`](https://keras.io/api/layers/reshaping_layers/up_sampling2d/)

In [ ]:
# EfficientNetV2B0 pré-entraîné sur ImageNet, sans sa tête de classification, gelé
modele_base = keras.applications.EfficientNetV2B0(
  include_top=False,
  weights="imagenet",
  input_shape=(dim_entree, dim_entree, nb_canaux),
)
modele_base.trainable = False
modele_base.output_shape

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `entrees = keras.Input(shape=(dim_entree, dim_entree, nb_canaux))`, puis `x = modele_base(entrees)` : des cartes de 16 × 16 × 1 280
- `layers.Conv2D(512, 1, activation="relu")` : réduit d'abord les 1 280 canaux à 512
- Cinq fois de suite : `layers.UpSampling2D(2)`, qui double la résolution, puis `layers.Conv2D(filtres, 3, padding="same", activation="relu")`, avec 256, 128, 64, 32 puis 16 filtres
- `layers.Dropout(0.1)`, puis `layers.Conv2D(1, 3, padding="same", activation="sigmoid")` : la probabilité de forêt de chaque pixel
- `keras.Model(entrees, sorties, name="no_skip")`, puis les mêmes appels que pour le UNet : `entraine`, `evaluate`, `predict`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
# Pixels de 0 à 255
entrees = keras.Input(shape=(dim_entree, dim_entree, nb_canaux))
x = modele_base(entrees)  # 16 × 16 × 1280
x = layers.Conv2D(512, 1, activation="relu")(x)
for filtres in [256, 128, 64, 32, 16]:
  x = ...  # résolution doublée
  x = layers.Conv2D(filtres, 3, padding="same", activation="relu")(x)
x = layers.Dropout(0.1)(x)
sorties = ...
modele_sans_saut = keras.Model(entrees, sorties, name="no_skip")

entraine(
  modele_sans_saut, 1e-3, "meilleur_sans_saut.keras", nb_epoques=nb_epoques_pretraine
)
scores = ...
resultats_test["EfficientNetV2B0 sans connexions de saut"] = scores
predictions = modele_sans_saut.predict(X)[..., 0]
for x, y, p in zip(X, Y, predictions):
  affiche_echantillon(x, y, p)
```

### Solution

In [ ]:
# Pixels de 0 à 255
entrees = keras.Input(shape=(dim_entree, dim_entree, nb_canaux))
x = modele_base(entrees)  # 16 × 16 × 1280
x = layers.Conv2D(512, 1, activation="relu")(x)
# Décodeur : de 16 × 16 à 512 × 512, en doublant 5 fois la résolution, sans
# rien recevoir des couches intermédiaires de l'encodeur
for filtres in [256, 128, 64, 32, 16]:
  x = layers.UpSampling2D(2)(x)
  x = layers.Conv2D(filtres, 3, padding="same", activation="relu")(x)
x = layers.Dropout(0.1)(x)
sorties = layers.Conv2D(1, 3, padding="same", activation="sigmoid")(x)
modele_sans_saut = keras.Model(entrees, sorties, name="no_skip")
modele_sans_saut.summary()

entraine(
  modele_sans_saut, 1e-3, "meilleur_sans_saut.keras", nb_epoques=nb_epoques_pretraine
)
scores = modele_sans_saut.evaluate(donnees_test, return_dict=True)
resultats_test["EfficientNetV2B0 sans connexions de saut"] = scores
print(resultats_test)

# Les mêmes tuiles de test que pour le UNet
predictions = modele_sans_saut.predict(X)[..., 0]
for x, y, p in zip(X, Y, predictions):
  affiche_echantillon(x, y, p)

Comparez ses masques à ceux du UNet, et le détail des contours à celui de la cible : le décodeur reconstruit les contours à partir de cartes de 16 × 16 seulement, où un pixel couvre 32 × 32 pixels de la tuile. L'encodeur pré-entraîné reconnaît la forêt, mais sans connexions de saut, le décodeur sait moins précisément *où* passent ses bords : cela se voit sur les lisières, pas forcément dans l'IoU.

## UNet pré-entraîné

Combinons les deux : un vrai UNet, avec connexions de saut, dont l'encodeur est le même EfficientNetV2B0 pré-entraîné (slide « UNet avec un encodeur pré-entraîné »). Le décodeur est fourni plus bas : c'est l'encodeur qu'il vous reste à construire.

Pour une tuile de 512 × 512, l'encodeur rend cinq sorties : celles de quatre couches intermédiaires de `modele_base`, à 256, 128, 64 et 32 pixels de côté, que le décodeur reçoit par les connexions de saut, puis le fond du « U », `modele_base.output`, à 16 × 16. À chaque résolution, prenez la dernière couche de `modele_base` qui la produit : sa sortie est la plus élaborée de cette résolution, juste avant que le réseau ne la divise par deux.

Trouvez les noms de ces quatre couches en parcourant `modele_base.layers` (`modele_base.summary()` les liste aussi, une par ligne), construisez `encodeur`, un `keras.Model` à cinq sorties, et gelez-le : seul le décodeur apprend d'abord. Affichez les formes de ses sorties pour vérifier leurs résolutions.

In [ ]:
# EfficientNetV2B0 pré-entraîné sur ImageNet, sans sa tête de classification
modele_base = keras.applications.EfficientNetV2B0(
  include_top=False,
  weights="imagenet",
  input_shape=(dim_entree, dim_entree, nb_canaux),
)
# Nombre de couches
len(modele_base.layers)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `for couche in modele_base.layers:` parcourt les couches dans l'ordre du calcul ; `forme = couche.output.shape` vaut `(None, côté, côté, canaux)` pour une carte de caractéristiques, mais n'a que deux dimensions pour quelques couches (celles du *squeeze-and-excitation*), à écarter : `if len(forme) == 4:`
- `dernieres[forme[1]] = couche.name`, dans cette boucle : à la fin, le dictionnaire `dernieres` donne la dernière couche de chaque côté
- `[modele_base.get_layer(nom).output for nom in noms]` : les sorties de ces couches
- `keras.Model(modele_base.input, caracteristiques + [modele_base.output], name="encodeur")` : l'encodeur, ses quatre sorties intermédiaires puis le fond du « U »
- `encodeur.trainable = False` gèle l'encodeur
- Formes attendues : `(None, 256, 256, 16)`, `(None, 128, 128, 32)`, `(None, 64, 64, 192)`, `(None, 32, 32, 672)` et `(None, 16, 16, 1280)`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
dernieres = {}  # côté → nom de la dernière couche de ce côté
for couche in modele_base.layers:
  forme = couche.output.shape
  if len(forme) == 4:  # (None, côté, côté, canaux)
    dernieres[...] = couche.name
noms = [dernieres[cote] for cote in (256, 128, 64, 32)]
print(noms)
caracteristiques = [... for nom in noms]
encodeur = keras.Model(
  modele_base.input, caracteristiques + [modele_base.output], name="encodeur"
)
encodeur.trainable = ...
[sortie.shape for sortie in encodeur.outputs]
```

### Solution

In [ ]:
# Dernière couche de chaque résolution : sa sortie est la plus élaborée de ce côté,
# juste avant que le réseau ne divise la résolution par deux
dernieres = {}
for couche in modele_base.layers:
  forme = couche.output.shape
  # Cartes de caractéristiques seulement : (None, côté, côté, canaux)
  if len(forme) == 4:
    dernieres[forme[1]] = couche.name
noms = [dernieres[cote] for cote in (256, 128, 64, 32)]
print(noms)

caracteristiques = [modele_base.get_layer(nom).output for nom in noms]
# Quatre sorties intermédiaires, puis le fond du « U », à 16 × 16
encodeur = keras.Model(
  modele_base.input, caracteristiques + [modele_base.output], name="encodeur"
)
# Gelé : seul le décodeur apprend d'abord
encodeur.trainable = False
[sortie.shape for sortie in encodeur.outputs]

### Décodeur et entraînement

Les couches trouvées à 64 et 32 pixels sont des `…_expand_activation` : l'expansion du premier bloc de la résolution suivante, calculée juste avant que ce bloc ne divise la résolution par deux. La slide prend plutôt la sortie du dernier bloc de chaque résolution (`block3b_add`, `block5e_add`), qui a moins de canaux : deux choix courants, qui donnent au décodeur les mêmes résolutions.

La cellule suivante fournit le décodeur : à chaque étape, il double la résolution, concatène la sortie de l'encodeur de même résolution, puis applique le `bloc_convolution` du premier UNet ; un dernier doublement le ramène à 512 × 512. Elle entraîne ensuite le modèle comme le précédent, avec un pas de 1e-3 (seul le décodeur, parti de zéro, apprend), le sauvegarde dans `meilleur_unet_pretraine.keras`, et range ses scores de test dans `resultats_test["UNet pré-entraîné, encodeur gelé"]`. Comparez ses prédictions sur `X` à celles des deux modèles précédents.

In [ ]:
# Pixels de 0 à 255
entrees = keras.Input(shape=(dim_entree, dim_entree, nb_canaux))
*sauts, x = encodeur(entrees)
# Décodeur : à chaque étape, la résolution est doublée, puis la sortie de
# l'encodeur de même résolution est concaténée (connexion de saut)
for filtres, saut in zip([256, 128, 64, 32], reversed(sauts)):
  x = layers.Conv2DTranspose(filtres, 2, strides=2)(x)
  x = bloc_convolution(layers.Concatenate()([x, saut]), filtres)
x = layers.Conv2DTranspose(16, 2, strides=2, activation="relu")(x)  # 512 × 512
sorties = layers.Conv2D(1, 1, activation="sigmoid")(x)
unet = keras.Model(entrees, sorties, name="pretrained_unet")
unet.summary()

# Seul le décodeur, parti de zéro, apprend : un pas plus grand que pour le UNet
entraine(unet, 1e-3, "meilleur_unet_pretraine.keras", nb_epoques=nb_epoques_pretraine)
scores = unet.evaluate(donnees_test, return_dict=True)
resultats_test["UNet pré-entraîné, encodeur gelé"] = scores
print(resultats_test)

predictions = unet.predict(X)[..., 0]
for x, y, p in zip(X, Y, predictions):
  affiche_echantillon(x, y, p)

### Fine-tuning de l'encodeur

Une fois le décodeur entraîné, dégelez l'encodeur, et poursuivez l'apprentissage de tout le réseau pendant `nb_epoques_affinage` epochs, avec un pas de 1e-5 : les poids de l'encodeur sont déjà bons, un grand pas les détruirait (slide « Fine-tuning : dégeler le backbone »). Deux pièges de Keras :

- les couches `BatchNormalization` de l'encodeur doivent rester gelées : sinon, elles recalculent leurs statistiques sur nos batchs de 4 tuiles infrarouges, bien différentes de celles d'ImageNet, et dégradent l'encodeur ;
- un changement de `trainable` ne compte qu'après un nouveau `compile`, que `entraine` fait.

Sauvegardez le modèle dans `meilleur_unet_affine.keras`, puis rangez ses scores de test dans `resultats_test["UNet pré-entraîné, fine-tuné"]`.

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `encodeur.trainable = True` dégèle toutes ses couches, `BatchNormalization` comprises
- `for couche in encodeur.layers:`, puis `if isinstance(couche, layers.BatchNormalization):` et `couche.trainable = False` : les regèle une à une
- `entraine(unet, 1e-5, "meilleur_unet_affine.keras", nb_epoques=nb_epoques_affinage)` repart des poids de `unet`, ceux du meilleur décodeur

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
encodeur.trainable = True
for couche in encodeur.layers:
  if isinstance(couche, ...):
    couche.trainable = ...

entraine(unet, ..., "meilleur_unet_affine.keras", nb_epoques=nb_epoques_affinage)
scores = ...
resultats_test["UNet pré-entraîné, fine-tuné"] = scores
```

#### Solution

In [ ]:
# Encodeur dégelé, sauf ses BatchNormalization, qui gardent les statistiques
# d'ImageNet
encodeur.trainable = True
for couche in encodeur.layers:
  if isinstance(couche, layers.BatchNormalization):
    couche.trainable = False

# Un pas 100 fois plus petit : ajuster les poids pré-entraînés sans les détruire
entraine(unet, 1e-5, "meilleur_unet_affine.keras", nb_epoques=nb_epoques_affinage)
scores = unet.evaluate(donnees_test, return_dict=True)
resultats_test["UNet pré-entraîné, fine-tuné"] = scores
print(resultats_test)

## Comparaison des modèles

La cellule suivante compare les scores de test des quatre modèles, ainsi que l'exactitude d'un modèle qui répondrait toujours la classe la plus fréquente des tuiles de test. Quel modèle obtient la meilleure IoU ? Les écarts entre les autres sont-ils assez grands pour les classer ? Pourquoi les écarts d'exactitude entre les modèles sont-ils plus faibles que les écarts d'IoU ?

In [ ]:
# Part des pixels de forêt dans la partie recadrée des tuiles de test
foret = parts.mean()
print(
  "Exactitude en répondant toujours la classe la plus fréquente :",
  f"{max(foret, 1 - foret):.3f}",
)
pandas.DataFrame(resultats_test).T[["iou", "accuracy", "loss"]].round(3)

### Solution

Le UNet pré-entraîné et fine-tuné arrive en tête à chaque exécution : il combine un encodeur qui a appris à reconnaître des textures sur ImageNet, les connexions de saut, qui gardent les contours précis, et un réglage de tout le réseau sur nos images infrarouges. Les trois autres modèles sont à un ou deux points d'IoU les uns des autres, et leur ordre change d'une exécution à l'autre : sur cette tâche facile (forêt ou non-forêt), ces écarts sont de l'ordre de la variation d'un entraînement à l'autre. Ils ne permettent donc pas de dire si les connexions de saut, le pré-entraînement ou la quantité de données font la différence. Pour les départager, il faudrait répéter chaque entraînement avec plusieurs graines et comparer les moyennes, ou regarder les tuiles à lisières affichées plus haut, où les erreurs se voient.

L'exactitude compte tous les pixels, dont la grande majorité est facile : l'intérieur des zones de forêt ou de non-forêt. Elle se lit mal sans l'exactitude du modèle qui répond toujours la classe la plus fréquente : tous les modèles la dépassent largement, ce qui montre seulement qu'ils ont appris quelque chose, et leurs valeurs, serrées près de 1, cachent comment ils se trompent. L'IoU ne compte que les pixels que la prédiction ou la cible disent de la forêt : chaque pixel de forêt manqué ou en trop y pèse davantage, et elle étale donc un peu plus les écarts que l'exactitude. Elle les rend plus lisibles, sans les rendre significatifs pour autant.